In [1]:
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

Loading Files

In [2]:
train_data = pd.read_csv("../data/train.csv")
test_data = pd.read_csv("../data/test.csv")
sample_submission = pd.read_csv("../data/sample_submission.csv")

In [3]:
print("Train data:", train_data.shape)
print("Test data:", test_data.shape)
print("Sample submission:", sample_submission.shape)

Train data: (594194, 21)
Test data: (254655, 20)
Sample submission: (254655, 2)


Separate id, input features, and target

In [4]:
train_ids = train_data["id"]
test_ids = test_data["id"]

customer_features = train_data.drop(columns=["id", "Churn"])
test_features = test_data.drop(columns=["id"])

churn_target = train_data["Churn"].map({
    "No": 0,
    "Yes": 1
})

Check the new feature and target shapes

In [5]:
print("Customer feature data:", customer_features.shape)
print("Target data:", churn_target.shape)
print("Test feature data:", test_features.shape)

Customer feature data: (594194, 19)
Target data: (594194,)
Test feature data: (254655, 19)


Checking whether the target converted correctly

In [6]:
print(churn_target.value_counts())
print("\nPercentage:")
print((churn_target.value_counts(normalize=True) * 100).round(2))

Churn
0    460377
1    133817
Name: count, dtype: int64

Percentage:
Churn
0    77.48
1    22.52
Name: proportion, dtype: float64


Numerical and Categorical Columns

In [7]:
numeric_features = customer_features.select_dtypes(include=["int64", "float64"]).columns.tolist()
categorical_features = customer_features.select_dtypes(include=["object"]).columns.tolist()

print("Numeric features:")
print(numeric_features)

print("\nCategorical features:")
print(categorical_features)

Numeric features:
['SeniorCitizen', 'tenure', 'MonthlyCharges', 'TotalCharges']

Categorical features:
['gender', 'Partner', 'Dependents', 'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod']


Split training and Validation data

In [8]:
X_train, X_valid, y_train, y_valid = train_test_split(
    customer_features,
    churn_target,
    test_size=0.20,
    random_state=31,
    stratify=churn_target
)

Checking split size and churn balance

In [9]:
print("X_train:", X_train.shape)
print("X_valid:", X_valid.shape)

print("\nTraining target percentage:")
print((y_train.value_counts(normalize=True) * 100).round(2))

print("\nValidation target percentage:")
print((y_valid.value_counts(normalize=True) * 100).round(2))

X_train: (475355, 19)
X_valid: (118839, 19)

Training target percentage:
Churn
0    77.48
1    22.52
Name: proportion, dtype: float64

Validation target percentage:
Churn
0    77.48
1    22.52
Name: proportion, dtype: float64


Preprocessing Setup

In [10]:
preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numeric_features),
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features)
    ]
)

Testing the Preprocessing

In [11]:
X_train_ready = preprocessor.fit_transform(X_train)
X_valid_ready = preprocessor.transform(X_valid)

print("Prepared training data:", X_train_ready.shape)
print("Prepared validation data:", X_valid_ready.shape)

Prepared training data: (475355, 45)
Prepared validation data: (118839, 45)


In [12]:
print(type(X_train_ready))

<class 'numpy.ndarray'>


In [13]:
print("Customer feature data shape:", customer_features.shape)

print("\nNumeric features:")
print(numeric_features)

print("\nCategorical features:")
print(categorical_features)

print("\nTraining target percentage:")
print((y_train.value_counts(normalize=True) * 100).round(2))

print("\nValidation target percentage:")
print((y_valid.value_counts(normalize=True) * 100).round(2))

print("\nPrepared training data shape:", X_train_ready.shape)
print("Prepared validation data shape:", X_valid_ready.shape)

Customer feature data shape: (594194, 19)

Numeric features:
['SeniorCitizen', 'tenure', 'MonthlyCharges', 'TotalCharges']

Categorical features:
['gender', 'Partner', 'Dependents', 'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod']

Training target percentage:
Churn
0    77.48
1    22.52
Name: proportion, dtype: float64

Validation target percentage:
Churn
0    77.48
1    22.52
Name: proportion, dtype: float64

Prepared training data shape: (475355, 45)
Prepared validation data shape: (118839, 45)
